In [ ]:
# Векторный поиск с помощью sqlitesearch — постоянный векторный 
# поиск на основе SQLite.

# Метод приблизительного поиска ближайшего соседа (ANN) использует 
# упрощенный подход. Вместо сравнения со всеми возможными вариантами,
# он сначала сужает область поиска до региона вероятных совпадений.
# Затем он оценивает результаты только в пределах этого региона. 
# Возможно, он пропустит абсолютно наилучшее совпадение, но 
# результаты все равно будут хорошими, и это намного быстрее.

# sqlitesearch - выполняет векторный поиск с помощью своего 
# VectorSearchIndexкласса. Он хранит векторы в SQLite, 
# реальной дисковой базе данных, и использует стратегии 
# искусственных нейронных сетей для извлечения данных. 
# Поскольку данные хранятся на диске, один процесс может 
# записывать векторы, а другой — считывать их обратно.

In [13]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RAGBase

assistant = RAGBase( index = index, llm_client = openai_client )

In [ ]:
class RAGVector(RAGBase):
  def __init__(self, embedder, **kwargs):
    super().__init__(**kwargs)
    self.embedder = embedder

  def search(self, query, num_results = 5):
    query_vector = self.embedder.encode(query)
    filter_dict = {"course": self.course}

    return self.index.search(
      query_vector,
      num_results = num_results,
      filter_dict = filter_dict
    )

In [18]:
texts = []

for doc in documents:
  text = doc["question"] + " " + doc["answer"]
  texts.append(text)

In [19]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
  batch = texts[i:i + batch_size]
  batch_vectors = model.encode(batch)
  vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1400

In [20]:
import numpy as np

X = np.array(vectors)

In [21]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [22]:
vector_assistant = RAGVector(
  embedder = model,
  index = vindex,
  llm_client = openai_client,
)

In [ ]:
# ___ Начало урока ___

# sqlitesearch поддерживает три режима работы нейронных сетей:
# - lsh(по умолчанию): до 100 тыс. векторов, случайные проекции гиперплоскостей
# - ivf: 10K-500K векторов, кластеризация методом K-средних
# - hnsw: 10K-1M+ векторов, граф близости (наивысшая полнота)

# Все режимы используют двухэтапный поиск: приблизительный поиск кандидатов,
# а затем точное переранжирование по косинусному сходству.


In [9]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
  keyword_fields=["course"],
  mode="ivf",
  db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
vs_index.fit(vectors, documents)

# В отличие от minsearch, этот файл постоянно находится на диске. 
# Вы можете выполнить поиск сразу после индексирования или повторно
# открыть индекс позже без повторного индексирования. 

In [24]:
# Поиск работает также как и minsearch - сначала кодируем запрос в вектор. 
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results = 5)

In [25]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [ ]:
# Когда закончите работу с индексом:

vs_index.close()